In [12]:
import os
import pandas as pd
import toml

config = toml.load('config.toml')

Data Request Info:
4/21/2026
Steffen Coenen, DKS
EVSE Regional Plan

SoundCast trip volumes:
- For scaling purposes, we'd like to request the following:
  - Volume of trips ending in each respective census tract.
  - Volume of trips intersecting with each respective census tract.

SoundCast commute trip volumes by destination census tract:
- This should capture all trips with "Work" as the destination ending in each respective census tract.
SoundCast long-distance trip volumes by intersecting census tract:
- This should capture all trips above 100 miles in length that intersect with each respective census tract.

SoundCast TNC/taxi trip volumes:
- This should capture all trips made by TNC/taxi drivers that intersect with each respective census tract.

SoundCast dwell times:
- For this, we would like to request two distinct metrics:
  - The share of all dwell periods ("dwells") in each respective census tract that are 4 or more hours long. For example, if one tract has 1,000 daily between-trip dwells and 600 of them are 4 or more hours long, this share is 60%.
- The share of all dwells in each respective census tract that are up to 1 hour long. For example, if one tract has 1,000 daily between-trip dwells, and 150 of them are up to 1 hour long, the metric we'd like to use for this tract is 15%.

MOVES EV VMT %:
- We understand that PSRC uses MOVES to translate EV adoption rates (per WA state regulation) into air quality and GHG emission metrics. Does this workflow also include an EV VMT forecast (i.e. translating EV adoption % into an EV VMT %) or is EV adoption rate assumed to translate 1:1 into EV VMT uptick?

# Soundcast Trip Volumes

In [13]:
# Use trip tables to get total destinations by tract
# Use Emme API to get tables directly from Bank

In [14]:
# Load a parcel to tract lookup for aggregating Daysim records to tracts
# Load the sqlite lite table from the model
import os
import sqlite3
con = sqlite3.connect(
            os.path.join(
                config["model_run_dir_2023"],
                r"inputs/db/soundcast_inputs_2023.db",
            )
        )
# conn = os.path.join(config["model_run_dir_2023"], "inputs\db\soundcast_inputs_2023.db")

df_db = pd.read_sql(con=con,
                    sql="SELECT ParcelID, Census2020Tract, Census2020Block FROM parcel_2023_geography")

In [4]:
# Exclude parcels that do not have a tract assigned
print(f"Number of parcels without a tract assigned: {len(df_db[df_db['Census2020Tract'].isnull()])}. These parcels will be excluded from the analysis.")
df_db = df_db[df_db["Census2020Tract"].notnull()]

print(f"Number of parcels without a block assigned: {len(df_db[df_db['Census2020Block'].isnull()])}. These parcels will be excluded from the analysis.")
df_db = df_db[df_db["Census2020Block"].notnull()]

Number of parcels without a tract assigned: 94. These parcels will be excluded from the analysis.
Number of parcels without a block assigned: 0. These parcels will be excluded from the analysis.


In [5]:
taz_tract_df = pd.read_csv(os.path.join(config["working_dir"], 'taz_tract_lookup.csv'))

In [6]:
taz_area_share_df = pd.read_csv(os.path.join(config["working_dir"], 'taz_tract_area_share_lookup.csv'))

In [7]:
# Load Emme trip tables for a time period
# Use Emme API
import inro.emme.desktop.app as app
import inro.modeller as _m

filepath = f"N:/rtp_2026_2050/final_runs/sc_base_year_2023_final/soundcast/projects/LoadTripTables/LoadTripTables.emp"

desktop = app.start_dedicated(True, "cth", filepath)
data_explorer = desktop.data_explorer()
m = _m.Modeller(desktop) 
database_names = [
            database.title() for database in data_explorer.databases()
        ]

In [44]:
# Load Daily bank

database_title = "daily"
scenario_id = "1002"

if database_title in database_names:
    database_index = database_names.index(database_title)
    database = data_explorer.databases()[database_index]
    database.open()
    bank = m.emmebank

In [45]:
current_scenario = list(bank.scenarios())[0]
zonesDim = len(current_scenario.zone_numbers)
zones = current_scenario.zone_numbers

# load zones into a NumpyArray to index trips otaz and dtaz

# Create a dictionary lookup where key is the taz id and value is it's numpy index.
indexToZone = dict((index, zone) for index, zone in enumerate(zones))

In [46]:
# Get demand
sov_list = ["sov_inc1", "sov_inc2", "sov_inc3"]
hov2_list = ["hov2_inc1", "hov2_inc2", "hov2_inc3"]
hov3_list = ["hov3_inc1", "hov3_inc2", "hov3_inc3"]
tnc_list = ["tnc_inc1", "tnc_inc2", "tnc_inc3"]
truck_list = ["medium_truck", "heavy_truck"]

In [47]:
def calculate_tract_trips(matrix_data):

    """Calculate total destination trips by census tract from an Emme matrix.

    Trips for each TAZ are distributed proportionally to intersecting tracts
    based on the share of TAZ area within each tract.

    Args:
        matrix_data (pandas.DataFrame): DataFrame containing the Emme matrix data with TAZ indices as rows and columns.
    Returns:
        pandas.DataFrame: DataFrame with columns 'geoid20' (census tract ID) and 'total_dest_trips' (total trips to that tract).

    Note: some destination TAZs don't have a tract match. These are trips to external stations outside the modeled area. 
    """

    # Sum columns (destinations) of the matrix
    np_matrix = matrix_data.to_numpy()
    dest_sums = np_matrix.sum(axis=0)  # one value per destination TAZ index

    # Build a DataFrame of destination trips per TAZ
    dest_df = pd.DataFrame({
        'taz': [indexToZone[i] for i in range(len(dest_sums))],
        'trips': dest_sums
    })

    # Join TAZ -> census tract area-share lookup (one row per TAZ/tract intersection)
    dest_df = dest_df.merge(taz_area_share_df, on='taz', how='left')

    # Proportionally allocate trips to each intersecting tract
    dest_df['allocated_trips'] = dest_df['trips'] * dest_df['taz_area_share_in_tract']

    # Aggregate proportionally allocated trips by census tract
    tract_trips = dest_df.groupby('geoid20', as_index=False)['allocated_trips'].sum()
    tract_trips.rename(columns={'allocated_trips': 'total_dest_trips'}, inplace=True)

    return tract_trips


In [65]:
results_df = pd.DataFrame()

matrix_results = {}

for matrix_name in sov_list + hov2_list + hov3_list + tnc_list + truck_list:
    print(f"Processing matrix: {matrix_name}")
    matrix_id = bank.matrix(matrix_name).id
    emme_matrix = bank.matrix(matrix_id)
    matrix_data = emme_matrix.get_data()
    matrix_results[matrix_name] = matrix_data.to_numpy()  # Store the raw matrix data in the dictionary to be used later
    tract_trips = calculate_tract_trips(matrix_data)
    tract_trips['matrix_name'] = matrix_name  # Add a column to identify the matrix
    results_df = pd.concat([results_df, tract_trips], ignore_index=True)

Processing matrix: sov_inc1
Processing matrix: sov_inc2
Processing matrix: sov_inc3
Processing matrix: hov2_inc1
Processing matrix: hov2_inc2
Processing matrix: hov2_inc3
Processing matrix: hov3_inc1
Processing matrix: hov3_inc2
Processing matrix: hov3_inc3
Processing matrix: tnc_inc1
Processing matrix: tnc_inc2
Processing matrix: tnc_inc3
Processing matrix: medium_truck
Processing matrix: heavy_truck


In [66]:
# matrix_data.to_numpy()

In [68]:
# Export daily trip matrices with TAZ IDs to CSV files
passenger_veh_matrix = matrix_results["sov_inc1"] + matrix_results["sov_inc2"] + matrix_results["sov_inc3"] + matrix_results["hov2_inc1"] + matrix_results["hov2_inc2"] + matrix_results["hov2_inc3"] + matrix_results["hov3_inc1"] + matrix_results["hov3_inc2"] + matrix_results["hov3_inc3"]

for matrix_name, matrix_data in {"passenger_vehicle": passenger_veh_matrix,
                                 "medium_truck": matrix_results["medium_truck"],
                                 "heavy_truck": matrix_results["heavy_truck"]}.items():
    # Create a DataFrame from the matrix data
    matrix_df = pd.DataFrame(matrix_data, index=[indexToZone[i] for i in range(len(matrix_data))], columns=[indexToZone[i] for i in range(len(matrix_data))])
    
    # Save the matrix with zone IDs to a CSV file
    matrix_df.to_csv(os.path.join(config["working_dir"], f"{matrix_name}_daily_trip_table.csv"))



In [72]:
# Export distance skims

# Change to 7to8 period to get distance skims
database_title = "7to8"
scenario_id = "1002"

if database_title in database_names:
    database_index = database_names.index(database_title)
    database = data_explorer.databases()[database_index]
    database.open()
    bank = m.emmebank

for matrix_name, title in {"sov_inc2d": "passenger_vehicle", 
                           "medium_truckd": "medium_truck", 
                           "heavy_truckd": "heavy_truck"}.items():
    print(f"Processing matrix: {matrix_name}")
    matrix_id = bank.matrix(matrix_name).id
    emme_matrix = bank.matrix(matrix_id)
    matrix_data = emme_matrix.get_data().to_numpy() 
    
    matrix_df = pd.DataFrame(matrix_data, index=[indexToZone[i] for i in range(len(matrix_data))], columns=[indexToZone[i] for i in range(len(matrix_data))])
    matrix_df.to_csv(os.path.join(config["working_dir"], f"{title}_distance_skim.csv"))

Processing matrix: sov_inc2d
Processing matrix: medium_truckd
Processing matrix: heavy_truckd


In [ ]:
# Reorganize tract results so that each matrix is a column and rows are indexed by geoid20
sorted_results_df = results_df.pivot(index='geoid20', columns='matrix_name', values='total_dest_trips')
sorted_results_df

# Combine all sov, hov2, hov3, tnc into individual sov, hov, tnc categories
sorted_results_df['passenger_vehicle_total'] = sorted_results_df[sov_list + hov2_list + hov3_list].sum(axis=1)
sorted_results_df['tnc_total'] = sorted_results_df[tnc_list].sum(axis=1)

In [14]:
sorted_results_df[["passenger_vehicle_total", "tnc_total","medium_truck", "heavy_truck"]]

matrix_name,passenger_vehicle_total,tnc_total,medium_truck,heavy_truck
geoid20,,,,
5.303300e+10,4842.125952,20.316073,141.030547,0.000000
5.303300e+10,13697.573165,46.702418,260.918186,0.000000
5.303300e+10,6277.810517,26.489680,151.292420,0.000000
5.303300e+10,7767.508788,30.780178,166.371696,42.190784
5.303300e+10,7716.144248,20.215244,134.540696,0.000000
...,...,...,...,...
5.306105e+10,6162.323026,4.592076,173.862488,99.912356
5.306105e+10,14451.135903,18.478837,273.298196,38.002392
5.306105e+10,6560.971139,5.456079,155.315991,21.128484


In [15]:
# Sum up results by time of day or use 

In [16]:
# dest_sums.sum()

In [17]:
sorted_results_df[["passenger_vehicle_total", "tnc_total","medium_truck", "heavy_truck"]].to_csv(os.path.join(config["working_dir"], "trips_by_tract_destination.csv"))

# Work Trips

In [ ]:
for year in ["2023", "2035", "2050"]:
    print(year)
    df_trip = pd.read_csv(os.path.join(config[f"model_run_dir_{year}"], "outputs/daysim/_trip.tsv"), sep="\t")

    # Select work trips
    work_trips = df_trip[df_trip["dpurp"] == 1]

    # select work trips for people using SOV, HOV2, or HOV3, by tract. 
    # select driver trips only to ensure only the vehicle trips are considered.
    work_trips_by_veh = work_trips[(work_trips["mode"].isin([3,4,5])) & (work_trips["dorp"] == 1)]

    # Merge tract geoid to dpcl
    work_trips_by_veh = work_trips_by_veh.merge(df_db, left_on="dpcl", right_on="ParcelID", how="left")

    for geog in ["Census2020Tract", "Census2020Block"]:

        work_trips_by_tract = work_trips_by_veh.groupby(geog).size().reset_index(name=f"{geog}_work_trips_in_vehicle")
        work_trips_by_tract["geoid20"] = work_trips_by_tract[geog].astype("int64").astype("str")

        work_trips_by_tract.to_csv(os.path.join(config["working_dir"], f"work_trips_dest_by_{geog.lower()}_{year}.csv"), index=False)

In [36]:
work_trips_by_tract.head()

,Census2020Block,tract_work_trips_in_vehicle,geoid20
0,5.303300e+14,87,530330001011001
1,5.303300e+14,31,530330001011002
2,5.303300e+14,14,530330001011005
3,5.303300e+14,44,530330001012000
4,5.303300e+14,156,530330001013000


# (Alternative) Tours to Tracts
Calculate an alternative to trips_by_tract_destination that uses Daysim data instead of trip tables. 
The advantage of this approach is that we can filter only for trips that are not back to home, so they capture true destination

The downsides are that this only includes trips from people simulated by Daysim (regional residents) and excludes external travel

# Alternative Data

In [ ]:
for year in ["2023", "2035", "2050"]:
    print(year)
    df_trip = pd.read_csv(os.path.join(config[f"model_run_dir_{year}"], "outputs/daysim/_trip.tsv"), sep="\t")

    # Filter for trips not ending at home
    df_nonhome_trip = df_trip[df_trip["dpurp"] != 0]

    # select only vehicle trips
    df_nonhome_trip = df_nonhome_trip[df_nonhome_trip["mode"].isin([3,4,5]) & (df_nonhome_trip["dorp"] == 1)]

    # Merge tract geoid to dpcl
    df_nonhome_trip = df_nonhome_trip.merge(df_db, left_on="dpcl", right_on="ParcelID", how="left")

    for geog in ["Census2020Tract", "Census2020Block"]:
        nonhome_trips_by_tract = df_nonhome_trip.groupby(geog).size().reset_index(name=f"{geog}_nonhome_trips_in_vehicle")
        nonhome_trips_by_tract["geoid20"] = nonhome_trips_by_tract[geog].astype("int64").astype("str")

        nonhome_trips_by_tract.to_csv(os.path.join(config["working_dir"], f"nonhome_trips_dest_by_{geog.lower()}_{year}.csv"), index=False)

2023
2035
2050


In [ ]:

# This allows us to 

# Dwell Times

Q: current approach includes all trips including those returning home. Do we want to exclude these?

In [16]:
def compute_dwell_time(df, geog, time_break) -> pd.DataFrame:

    """Compute share of dwells above and below a time_break (hours) by geography from a Daysim trip table.
    Args:
        df (pandas.DataFrame): DataFrame containing the Daysim trip data with columns 'mode', 'dorp', 'endacttm', 'arrtm', and 'dtaz'.
        geog (str): Geographic level to group by, either 'Census2020Tract' or 'Census2020Block'.
        time_break (int): Time break in hours to classify dwells, e.g., 4 for 4+ hours and 1 for less than 1 hour.
    Returns:
        pandas.DataFrame: DataFrame with columns 'geoid20' (census tract ID), 'dwell_time_4plus_hours' (share of dwells 4+ hours), and 'dwell_time_less_than_1_hr' (share of dwells less than 1 hour).  
    """

    # The share of all dwell periods ("dwells") in each respective census tract that are 4 or more hours long. 
    # For example, if one tract has 1,000 daily between-trip dwells and 600 of them are 4 or more hours long, this share is 60%.
    mask = df["dwell_time"] >= time_break * 60
    df.loc[mask, f"dwell_time_{time_break}plus_hours"] = 1

    # The share of all dwells in each respective census tract that are up to 1 hour long. 
    # For example, if one tract has 1,000 daily between-trip dwells, and 150 of them are up to 1 hour long, the metric we'd like to use for this tract is 15%.
    mask = df["dwell_time"] < time_break * 60
    df.loc[mask, f"dwell_time_less_than_{time_break}_hr"] = 1

    # Calculate the share of dwells 4+ hours and less than 1 hour by tract
    # What percent of trips to each tract have a dwell time of 4 or more hours? What percent have a dwell time of less than 1 hour?
    dwell_stats_by_tract = df.groupby(geog).agg(
        total_dwells=pd.NamedAgg(column="dwell_time", aggfunc="count"),
        dwells_4plus_hours=pd.NamedAgg(column=f"dwell_time_{time_break}plus_hours", aggfunc="sum"),
        dwells_less_than_1_hr=pd.NamedAgg(column=f"dwell_time_less_than_{time_break}_hr", aggfunc="sum")
        ).reset_index()

    dwell_stats_by_tract[f"percent_dwell_time_{time_break}plus_hours"] = dwell_stats_by_tract["dwells_4plus_hours"] / dwell_stats_by_tract["total_dwells"]
    dwell_stats_by_tract[f"percent_dwell_time_less_than_{time_break}_hr"] = dwell_stats_by_tract["dwells_less_than_1_hr"] / dwell_stats_by_tract["total_dwells"]   

    return dwell_stats_by_tract[[geog, f"percent_dwell_time_{time_break}plus_hours", f"percent_dwell_time_less_than_{time_break}_hr", "total_dwells"]]

In [ ]:
# for year in ["2023", "2035", "2050"]:
for year in ["2023","2035", "2050"]:
    print(year)
    df_trip = pd.read_csv(os.path.join(config[f"model_run_dir_{year}"], "outputs/daysim/_trip.tsv"), sep="\t")

    # Merge tract and block info based on destinaton parcel
    df_trip = df_trip.merge(df_db[["ParcelID", "Census2020Tract", "Census2020Block"]], left_on="dpcl", right_on="ParcelID", how="left")

    for geog in ["Census2020Tract", "Census2020Block"]:

        df_trip[geog] = df_trip[geog].fillna(0).astype("int64").astype("str")

        # Dwell times computed as difference between end of activity time and arrival time at activity
        df_trip["dwell_time"] = df_trip["endacttm"] - df_trip["arrtm"]

        # For first analysis:
        # Select only vehicle trips
        df_trip_dwell = df_trip[df_trip["mode"].isin([3,4,5]) & (df_trip["dorp"] == 1)].copy()

        # For second analysis:
        # Select only trips not ending at home
        df_trip_dwell_nonhome = df_trip_dwell[df_trip_dwell["dpurp"] != 0].copy()

        # For third analysis"
        # Select only work trips
        df_trip_dwell_work = df_trip_dwell[df_trip_dwell["dpurp"] == 1].copy()

        for time_break in [2,4]:
            all_trips_dwell_time = compute_dwell_time(df_trip_dwell, geog=geog, time_break=time_break)

            nonhome_trips_dwell_time = compute_dwell_time(df_trip_dwell_nonhome, geog=geog, time_break=time_break)

            work_trips_dwell_time = compute_dwell_time(df_trip_dwell_work, geog=geog, time_break=time_break)

            all_trips_dwell_time.to_csv(os.path.join(config["working_dir"], f"{time_break}_hour_dwell_all_trips_{geog.lower()}_{year}.csv"), index=False)
            nonhome_trips_dwell_time.to_csv(os.path.join(config["working_dir"], f"{time_break}_hour_dwell_nonhome_trips_{geog.lower()}_{year}.csv"), index=False)
            work_trips_dwell_time.to_csv(os.path.join(config["working_dir"], f"{time_break}_hour_dwell_work_trips_{geog.lower()}_{year}.csv"), index=False)

2023
2035


In [2]:
year = 2023
df_trip = pd.read_csv(os.path.join(config[f"model_run_dir_{year}"], "outputs/daysim/_trip.tsv"), sep="\t")

In [10]:
(62/len(df_trip))*100

0.0003806369726443575

In [5]:
df_trip[df_trip["travdist"] > 100]

,id,tour_id,hhno,pno,day,tour,half,tseg,tsvid,opurp,...,deptm,arrtm,endacttm,travtime,travcost,travdist,vot,trexpfac,od,sov_ff_time
3184764,897371101,8973711,340651,2,1,1,1,1,0,0,...,261,398,891,137.730000,20.184,100.92,14.997243,1,3444-2127,11071
6444331,1730898101,17308981,745250,2,1,1,1,1,0,0,...,383,487,1049,104.899405,7.670,101.79,6.339470,1,1871-3694,5595
7378912,1951557101,19515571,835629,2,1,1,1,1,0,0,...,296,410,938,114.863750,7.670,103.15,9.433986,1,1679-3691,6127
12198880,-1071885195,32230821,1307323,2,1,1,1,1,0,0,...,268,405,905,137.000000,23.138,104.49,46.761792,1,1995-2808,11682
13314470,-764548195,35304191,1431903,2,1,1,1,1,0,0,...,334,453,642,119.360000,10.170,106.76,8.945866,1,2319-3692,7566
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15568270,-169718145,41252491,1647259,2,1,1,2,1,0,4,...,626,750,758,124.290000,20.452,102.26,16.703187,1,1684-3688,7236
15638009,-151669145,41432981,1654843,1,1,1,2,1,0,3,...,855,996,1153,141.590000,21.470,107.35,4.153135,1,2181-3492,8676
15690370,-97536845,41974304,1675812,2,1,4,2,1,0,5,...,1147,1271,179,124.790000,20.204,101.02,9.897718,1,1649-3486,7952
15698249,-91469145,42034981,1678228,4,1,1,2,1,0,3,...,723,861,179,138.900000,20.636,103.18,7.069501,1,2250-3507,8177


In [77]:
nonhome_trips_dwell_time.describe()

,percent_dwell_time_4plus_hours,percent_dwell_time_less_than_1_hr
count,43029.000000,43029.000000
mean,0.073289,0.601432
std,0.098795,0.155093
min,0.000000,0.000000
25%,0.000000,0.524324
50%,0.043478,0.607143
75%,0.103448,0.678571
max,1.000000,1.000000
